<a href="https://colab.research.google.com/github/sotesh1516/Transformer-Attention_Is_All_You_Need/blob/main/Transformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import torch

The following are part of tokenization, converting text into discreet IDs.

In [ ]:
def build_token_to_id_vocab(sentences, specials=('<pad>', '<bos>', '<eos>', '<unk>')):
    # build a token-to-id dict with specials first, then corpus tokens in first-seen order.
    vocab_dict = {}
    global_id_counter = 0
    for sp in specials:
        vocab_dict[sp] = global_id_counter
        global_id_counter+=1
    for w in sentences:
        w_arr = w.split()
        for t in w_arr:
            if t not in vocab_dict:
                vocab_dict[t] = global_id_counter
                global_id_counter+=1

    return vocab_dict

def build_id_to_token_vocab(token_to_id):
    # build the inverse id-to-token dictionary from token_to_id
    id_to_token = {}
    for token in token_to_id:
        id_to_token[token_to_id[token]] = token

    return id_to_token

def encode_sentence_to_ids(sentence, token_to_id, unk_token='<unk>'):
    # convert whitespace tokens of `sentence` to ids via `token_to_id`, using `unk_token`'s id for OOV
    int_token_id = []
    whitespace_token = sentence.split()
    for t in whitespace_token:
        if t not in token_to_id:
            int_token_id.append(token_to_id[unk_token])
        else:
            int_token_id.append(token_to_id[t])

    return int_token_id

def decode_ids_to_tokens(ids, id_to_token):
    # map each id in ids to its token string via id_to_token and return the list
    tokens = []
    for id in ids:
        tokens.append(id_to_token[id])
    return tokens

def pad_id_sequence(ids, max_len, pad_id):
    # return a list of length exactly max_len, padding with pad_id or truncating.
    if max_len > len(ids):
        return ids + [pad_id for i in range(max_len - len(ids))]
    return ids[:max_len]

def stack_padded_sequences_to_batch(padded_sequences):
    """Stack a list of equal-length padded id sequences into a 2D LongTensor batch."""
    # stack padded id sequences into a (B, L) torch.long tensor
    return torch.tensor(padded_sequences).long()

Embeddings and Positional Encoding

In [ ]:
def scale_embeddings_by_sqrt_d_model(embeddings, d_model):
    """Scale a token embedding tensor by sqrt(d_model)."""
    # rescale embeddings by sqrt(d_model) as in the original Transformer paper
    return embeddings * math.sqrt(d_model)

def compute_positional_div_term(d_model):
    """
    Note:
    - Using several different frequencies gives each position a distinguishable vector pattern.
    - Assign each feature channel(dimension) its own angular frequency
    - Here we are producing pair-wise freq
        - frequency for pair 0 (dim 0 and 1)
        - frequency for pair 1
        - frequency for pair 2
    - Finally we can do pos * 1000^(-2i/d_model) -> this is the current step
    """
    # return a 1D FloatTensor of length d_model // 2 holding the sinusoidal frequency divisors
    output_len = d_model // 2
    i = torch.arange(output_len, dtype=torch.float32)
    freq = 10000 ** (-2.0 * i / d_model)
    return freq

def build_position_index_column(max_len):
    """
    Return a (max_len, 1) float tensor of [0, 1, ..., max_len-1].

    The sinusoidal positional encoding evaluates sine and cosine at every (position, frequency) pair.
    """
    # build a column vector of position indices from 0 to max_len-1
    pos_indices = torch.empty(max_len,1)
    for i in range(max_len):
        pos_indices[i] = torch.tensor(i).float()
    return pos_indices

def fill_even_indices_with_sin(pe, position, div_term):
    """Fill even feature indices of pe with sin(position * div_term)."""
    # write sin(position * div_term) into the even-indexed columns of pe and return it
    for i in range(len(pe)):
        for j in range(len(pe[0])):
            if j % 2 == 0: #even dimension
                current_dim_freq = j // 2
                pe[i, j] = torch.sin(position[i] * div_term[current_dim_freq])

    return pe

def fill_odd_indices_with_cos(pe, position, div_term):
    # TODO: fill the odd-indexed columns of pe with cos(position * div_term)
    for i in range(len(pe)):
        for j in range(len(pe[0])):
            if j % 2 != 0: #odd dimension
                current_dim_freq = j // 2
                pe[i, j] = torch.cos(position[i] * div_term[current_dim_freq])

    return pe